# Conversión de artículos científicos en PDF a Markdown

Este notebook toma artículos científicos en formato **PDF** y los convierte a **Markdown** (`.md`),
conservando en la medida de lo posible la estructura del documento: títulos, secciones, párrafos,
tablas e imágenes.

El caso de uso principal es procesar artículos científicos (por ejemplo, papers sobre lluvia,
movimientos en masa, sistemas de alerta temprana, radar meteorológico, etc.) para poder:

- Leerlos y anotarlos más fácilmente en texto plano / Markdown.
- Indexarlos o resumirlos con herramientas de IA (LLMs) que trabajan mejor con texto plano.
- Incluir fragmentos convertidos en reportes técnicos (Overleaf/LaTeX admite Markdown vía `pandoc`).

**Librería usada:** [`PyMuPDF`](https://pymupdf.readthedocs.io/) (paquete `pymupdf`, antes conocido
como `fitz`). Se usa directamente en vez de la utilidad de más alto nivel `pymupdf4llm` porque esta
última **requiere Python 3.10 o superior**; el conversor de este notebook está escrito a mano sobre
`PyMuPDF` puro para que funcione también en **Python 3.8**.

La conversión funciona así, página por página:

1. Se detectan **tablas** con `page.find_tables()` y se exportan directamente a Markdown.
2. El resto del **texto** se agrupa en bloques y se compara el tamaño de fuente de cada bloque contra
   el tamaño de fuente más común del documento (el "cuerpo de texto"): los bloques con letra más
   grande se convierten en encabezados (`#`, `##`, `###`...), y los bloques en negrita y cortos se
   resaltan en **negrita**.
3. Se extraen las **imágenes** de cada página como archivos aparte y se insertan como
   `![](ruta)` en el lugar correspondiente.
4. Todos los elementos de una página (texto, tablas, imágenes) se ordenan por su posición vertical
   para respetar el orden de lectura.

> Nota: si un PDF es un **escaneo** (imágenes de páginas, sin texto real embebido), este método no
> extraerá texto útil. Para esos casos se necesita OCR — ver la última sección del notebook.


## 1. Instalación de dependencias

Solo es necesario ejecutar esta celda una vez (o cuando cambies de entorno).

Se fija `PyMuPDF<1.24.12` a propósito: esa es la última serie de versiones que sigue publicando
ruedas (`wheels`) para **Python 3.8**. Si ya usas Python 3.9+ puedes quitar esa restricción y usar
la versión más reciente sin problema.


In [ ]:
# Instalación de dependencias
# (ajusta o quita el tope de versión si usas Python 3.9 o superior)
%pip install -q "PyMuPDF<1.24.12" tqdm


## 2. Importar librerías


In [ ]:
from collections import Counter
from pathlib import Path

import pymupdf  # antes conocido como "fitz"
from tqdm.auto import tqdm


## 3. Configuración de rutas

- `INPUT_DIR`: carpeta donde están los PDF de los artículos científicos.
- `OUTPUT_DIR`: carpeta donde se guardarán los `.md` generados.
- `IMAGES_DIR`: carpeta donde se guardarán las imágenes/figuras extraídas de cada PDF (opcional).

Ajusta estas rutas según la estructura de tu proyecto.


In [ ]:
INPUT_DIR = Path("pdfs")            # carpeta con los PDF de entrada
OUTPUT_DIR = Path("markdown")       # carpeta donde se guardarán los .md
IMAGES_DIR = Path("markdown/imagenes")  # carpeta para las figuras extraídas

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

pdf_files = sorted(INPUT_DIR.glob("*.pdf"))
print("Se encontraron {} archivo(s) PDF en '{}':".format(len(pdf_files), INPUT_DIR))
for f in pdf_files:
    print(" -", f.name)


## 4. Funciones auxiliares para detectar encabezados

- `_tamanos_de_fuente`: recorre todo el documento y cuenta cuántos caracteres hay de cada tamaño de
  fuente. El tamaño más frecuente se asume como el "cuerpo de texto" normal del artículo.
- `_nivel_encabezado`: dado el tamaño de un bloque de texto, decide si es un encabezado (`#`, `##`,
  `###`, `####`) y de qué nivel, según qué tan grande es respecto al cuerpo de texto.
- `_bbox_solapa`: evita duplicar contenido que ya quedó capturado dentro de una tabla.


In [ ]:
def _tamanos_de_fuente(doc):
    """Cuenta caracteres por tamaño de fuente en todo el documento.

    Devuelve (tamano_cuerpo, tamanos_mayores) donde `tamano_cuerpo` es el tamaño más frecuente
    (el texto normal del artículo) y `tamanos_mayores` es la lista de tamaños más grandes que el
    cuerpo, ordenados de mayor a menor (candidatos a encabezados).
    """
    contador = Counter()
    for page in doc:
        data = page.get_text("dict")
        for block in data["blocks"]:
            if block.get("type") != 0:  # 0 = bloque de texto
                continue
            for line in block["lines"]:
                for span in line["spans"]:
                    if span["text"].strip():
                        contador[round(span["size"])] += len(span["text"])

    if not contador:
        return 10, []

    tamano_cuerpo = contador.most_common(1)[0][0]
    tamanos_mayores = sorted((t for t in contador if t > tamano_cuerpo), reverse=True)
    return tamano_cuerpo, tamanos_mayores


def _nivel_encabezado(tamano, tamano_cuerpo, tamanos_mayores, max_nivel=4):
    """Devuelve el nivel de encabezado (1 a max_nivel) para un tamaño de fuente, o 0 si es texto normal."""
    if tamano <= tamano_cuerpo or not tamanos_mayores:
        return 0
    idx = min(range(len(tamanos_mayores)), key=lambda i: abs(tamanos_mayores[i] - tamano))
    return min(idx + 1, max_nivel)


def _bbox_solapa(bbox, otros_bboxes, umbral=0.6):
    """True si `bbox` se solapa en al menos `umbral` de su área con alguno de `otros_bboxes`."""
    x0, y0, x1, y1 = bbox
    area = max(1e-6, (x1 - x0) * (y1 - y0))
    for ox0, oy0, ox1, oy1 in otros_bboxes:
        ix0, iy0 = max(x0, ox0), max(y0, oy0)
        ix1, iy1 = min(x1, ox1), min(y1, oy1)
        if ix1 > ix0 and iy1 > iy0:
            interseccion = (ix1 - ix0) * (iy1 - iy0)
            if interseccion / area >= umbral:
                return True
    return False


## 5. Función principal: convertir un PDF a Markdown

Recorre cada página del PDF, extrae tablas, texto (clasificando encabezados y negritas) e imágenes,
y arma un único archivo Markdown respetando el orden de lectura de arriba hacia abajo.


In [ ]:
def pdf_a_markdown(pdf_path: Path, output_dir: Path, images_dir=None) -> Path:
    """Convierte un único PDF a un archivo Markdown.

    Parameters
    ----------
    pdf_path : Path
        Ruta al archivo PDF de entrada.
    output_dir : Path
        Carpeta donde se guardará el archivo .md resultante.
    images_dir : Path, opcional
        Si se indica, se extraen las imágenes del PDF hacia esa carpeta.

    Returns
    -------
    Path
        Ruta del archivo .md generado.
    """
    pdf_path = Path(pdf_path)
    output_dir = Path(output_dir)
    doc = pymupdf.open(str(pdf_path))

    tamano_cuerpo, tamanos_mayores = _tamanos_de_fuente(doc)

    partes_documento = []
    contador_imagenes = 0

    for num_pagina, page in enumerate(doc, start=1):
        elementos = []  # lista de (posicion_y, texto_markdown), para ordenar al final

        # --- 1) Tablas ---
        bboxes_tablas = []
        try:
            tablas = page.find_tables()
        except Exception:
            tablas = []
        for tabla in tablas:
            bboxes_tablas.append(tabla.bbox)
            try:
                texto_tabla = tabla.to_markdown()
            except Exception:
                filas = tabla.extract()
                texto_tabla = "\n".join(" | ".join(str(c or "") for c in fila) for fila in filas)
            elementos.append((tabla.bbox[1], texto_tabla))

        # --- 2) Texto (títulos, negritas, párrafos) ---
        data = page.get_text("dict")
        for block in data["blocks"]:
            if block.get("type") != 0:
                continue
            if _bbox_solapa(block["bbox"], bboxes_tablas):
                continue  # el contenido ya quedó capturado como tabla

            lineas_texto = []
            tamanos_bloque = []
            es_negrita = True
            for line in block["lines"]:
                texto_linea = "".join(span["text"] for span in line["spans"]).strip()
                if not texto_linea:
                    continue
                lineas_texto.append(texto_linea)
                for span in line["spans"]:
                    if not span["text"].strip():
                        continue
                    tamanos_bloque.append(round(span["size"]))
                    if "Bold" not in span.get("font", "") and not (span.get("flags", 0) & 2 ** 4):
                        es_negrita = False

            if not lineas_texto:
                continue

            texto_bloque = " ".join(lineas_texto)
            tamano_dominante = (
                Counter(tamanos_bloque).most_common(1)[0][0] if tamanos_bloque else tamano_cuerpo
            )
            nivel = _nivel_encabezado(tamano_dominante, tamano_cuerpo, tamanos_mayores)

            if nivel > 0:
                texto_md = "{} {}".format("#" * nivel, texto_bloque)
            elif es_negrita and len(texto_bloque.split()) <= 20:
                texto_md = "**{}**".format(texto_bloque)
            else:
                texto_md = texto_bloque

            elementos.append((block["bbox"][1], texto_md))

        # --- 3) Imágenes ---
        if images_dir is not None:
            images_dir = Path(images_dir)
            images_dir.mkdir(parents=True, exist_ok=True)
            for img_info in page.get_images(full=True):
                xref = img_info[0]
                try:
                    rects = page.get_image_rects(xref)
                    pos_y = rects[0].y0 if rects else 0
                    base_imagen = doc.extract_image(xref)
                    contador_imagenes += 1
                    nombre_imagen = "{}_p{}_img{}.{}".format(
                        pdf_path.stem, num_pagina, contador_imagenes, base_imagen["ext"]
                    )
                    ruta_imagen = images_dir / nombre_imagen
                    ruta_imagen.write_bytes(base_imagen["image"])
                    ruta_relativa = "{}/{}".format(images_dir.name, nombre_imagen)
                    elementos.append((pos_y, "![]({})".format(ruta_relativa)))
                except Exception:
                    continue  # algunas imágenes (máscaras, patrones) pueden fallar; se ignoran

        elementos.sort(key=lambda e: e[0])
        contenido_pagina = "\n\n".join(e[1] for e in elementos)
        if contenido_pagina.strip():
            partes_documento.append(contenido_pagina)

    doc.close()

    md_text = "\n\n".join(partes_documento) + "\n"
    output_path = output_dir / (pdf_path.stem + ".md")
    output_path.write_text(md_text, encoding="utf-8")
    return output_path


## 6. Limpieza opcional del Markdown

Los artículos científicos suelen traer "ruido" heredado del PDF: palabras cortadas con guion al
final de línea (por el layout de dos columnas) y saltos de línea de más. Esta función aplica
limpiezas simples y seguras; ajústala según lo que veas en tus propios documentos.


In [ ]:
import re


def limpiar_markdown(texto: str) -> str:
    """Aplica limpiezas simples a un texto Markdown extraído de un PDF científico."""

    # 1) Unir palabras cortadas por guion al final de línea, ej: "modelo-\nmiento" -> "modelomiento"
    texto = re.sub(r"(\w)-\n(\w)", r"\1\2", texto)

    # 2) Colapsar 3+ líneas en blanco seguidas a solo 2 (un párrafo de separación)
    texto = re.sub(r"\n{3,}", "\n\n", texto)

    # 3) Quitar espacios en blanco al final de cada línea
    texto = re.sub(r"[ \t]+\n", "\n", texto)

    return texto.strip() + "\n"


## 7. Procesamiento por lotes (batch)

Convierte todos los PDF encontrados en `INPUT_DIR`, aplica la limpieza y guarda el resultado en
`OUTPUT_DIR`. Si un archivo falla (por ejemplo, un PDF corrupto o protegido), se reporta el error y
se continúa con los siguientes.


In [ ]:
resultados = []

for pdf_path in tqdm(pdf_files, desc="Convirtiendo PDFs"):
    try:
        # Subcarpeta de imágenes por artículo, para no mezclar figuras de distintos PDFs
        img_dir = IMAGES_DIR / pdf_path.stem

        salida = pdf_a_markdown(pdf_path, OUTPUT_DIR, images_dir=img_dir)

        texto_limpio = limpiar_markdown(salida.read_text(encoding="utf-8"))
        salida.write_text(texto_limpio, encoding="utf-8")

        resultados.append((pdf_path.name, str(salida), "OK"))
    except Exception as e:
        resultados.append((pdf_path.name, None, "ERROR: {}".format(e)))

print("\nResumen:")
for nombre, salida, estado in resultados:
    print(" - {}: {}".format(nombre, estado))


## 8. Vista previa de un resultado

Muestra el Markdown generado para el primer PDF procesado exitosamente, renderizado dentro del
notebook.


In [ ]:
from IPython.display import Markdown, display

exitosos = [r for r in resultados if r[2] == "OK"]

if exitosos:
    _, ruta_md, _ = exitosos[0]
    contenido = Path(ruta_md).read_text(encoding="utf-8")
    print("Vista previa de: {}\n".format(ruta_md))
    display(Markdown(contenido[:3000]))
else:
    print("Todavía no hay archivos convertidos. Revisa INPUT_DIR y vuelve a ejecutar.")


## 9. Convertir un único PDF puntual

Celda de conveniencia para convertir un solo artículo sin correr todo el lote, por ejemplo cuando
acabas de descargar un paper nuevo.


In [ ]:
# Ejemplo de uso:
# ruta_pdf = INPUT_DIR / "nombre_del_articulo.pdf"
# salida = pdf_a_markdown(ruta_pdf, OUTPUT_DIR, images_dir=IMAGES_DIR / ruta_pdf.stem)
# Path(salida).write_text(limpiar_markdown(salida.read_text(encoding="utf-8")), encoding="utf-8")
# print("Guardado en:", salida)


## 10. Notas, límites y alternativas

- **Compatibilidad con Python 3.8:** este notebook usa `PyMuPDF` puro (import `pymupdf`, antes
  `fitz`) en vez de `pymupdf4llm`, porque esta última exige Python ≥3.10. Si en el futuro actualizas
  a Python 3.9+ puedes usar `pymupdf4llm.to_markdown(...)` como atajo (una sola línea) en lugar de la
  función `pdf_a_markdown` de este notebook, con resultados similares o mejores.

- **PDFs escaneados (sin texto real):** si `pdf_a_markdown` devuelve un archivo vacío o con muy poco
  texto, el PDF probablemente es una imagen escaneada. En ese caso se necesita OCR antes de convertir
  a Markdown, por ejemplo con [`ocrmypdf`](https://ocrmypdf.readthedocs.io/) (`pip install ocrmypdf`,
  requiere Tesseract instalado en el sistema) para generar primero un PDF con capa de texto, y luego
  correr este mismo notebook sobre ese PDF ya "OCRizado".

- **Ecuaciones matemáticas:** las ecuaciones (frecuentes en papers de hidrología/geotecnia)
  normalmente se extraen como texto plano o símbolos sueltos, sin la notación matemática original.
  Revisar manualmente esas secciones en el `.md` de salida.

- **Detección de encabezados por tamaño de fuente:** es una heurística. Si un artículo usa el mismo
  tamaño de letra para títulos y cuerpo (poco común, pero posible), los títulos no se detectarán como
  tales. En ese caso, ajusta manualmente el `.md` de salida o añade una regla extra basada en negrita.

- **Múltiples columnas:** `PyMuPDF` reporta los bloques de texto por posición; el ordenamiento por
  coordenada vertical (`y0`) funciona razonablemente bien en layouts de dos columnas, pero conviene
  revisar manualmente el `.md` de salida en artículos con figuras/tablas incrustadas entre columnas.

- **Uso posterior en LaTeX/Overleaf:** puedes convertir estos `.md` a `.tex` con `pandoc`:
  `pandoc articulo.md -o articulo.tex`.
